# Нейронные сети: основы

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Нейронные сети: основы».

## Научная цель

Полносвязная нейронная сеть (многослойный перцептрон) по примерам учится переводить набор входных величин в нужный ответ, выявляя нелинейные зависимости. Это базовый инструмент-аппроксиматор: исследователь подаёт пары «параметры образца — измеренное свойство», а сеть подбирает свои внутренние веса так, чтобы предсказывать ответ для новых случаев.

В этом блокноте мы на компактном табличном наборе обучим небольшую сеть решать задачу регрессии, разберём роль нормировки входов и темпа обучения и научимся читать кривые обучения. Блокнот исполняется на CPU за секунды.

## Интуиция за методом

Представьте настройку оркестра перед концертом. Дирижёр прослушивает результат и чуть подправляет каждого музыканта — кому-то чуть тише, кому-то чуть быстрее. После многих репетиций общее звучание сближается с нужным. Обучение нейронной сети устроено так же: сеть делает предсказание, сравнивает с правильным ответом, вычисляет, кто «виноват в ошибке», и чуть сдвигает свои внутренние веса. После сотен таких проходов по данным предсказания становятся точнее.

**Ключевые понятия, которые встретятся в блокноте:**

- **Признак** — один измеренный параметр наблюдения (столбец таблицы).
- **Нейрон** — элементарная вычислительная единица: берёт несколько чисел, умножает каждое на свой вес, суммирует и применяет нелинейную функцию.
- **Скрытый слой** — промежуточный слой нейронов между входом и выходом; чем их больше и чем они шире, тем сложнее зависимости может выучить сеть.
- **Функция активации (ReLU)** — нелинейность, которая позволяет сети учить сложные, незаписываемые формулой зависимости.
- **Эпоха** — один полный проход по всем обучающим данным; обучение идёт несколько десятков или сотен эпох.
- **Темп обучения (learning rate)** — насколько сильно веса меняются за один шаг; слишком большой — сеть «скачет» и не сходится, слишком малый — учится слишком медленно.
- **Переобучение** — сеть запомнила шум обучающих данных и плохо работает на новых. **Dropout** — случайное отключение части нейронов во время обучения, одна из мер против переобучения.
- **Нормировка входов** — обязательный шаг: признаки приводят к нулевому среднему и единичному разбросу, иначе обучение идёт неустойчиво.
- **Кривая обучения** — график потери (ошибки) по числу эпох; позволяет увидеть, учится ли сеть и нет ли переобучения.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор `diabetes` из `scikit-learn`: 442 пациента, 10 числовых физиологических признаков (возраст, индекс массы тела, кровяное давление и другие биохимические показатели). Целевая величина — индекс прогрессирования диабета через год после измерений.

Это задача **регрессии**: нужно предсказать непрерывную величину по 10 признакам. Ячейка ниже загружает данные, разбивает на обучающую и тестовую выборки, и стандартизует признаки.

**Почему нормировка обязательна для нейросети?** Нейроны суммируют входы с весами. Если один признак имеет значения 0.001–0.005, а другой — 500–5000, веса для них будут на разных порядках, и обучение пойдёт нестабильно. После стандартизации (среднее 0, разброс 1) все входы сопоставимы по масштабу.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

data = load_diabetes()
X, y = data.data.astype('float32'), data.target.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

# Нормировка входов: среднее 0, разброс 1 — обязательный шаг для сети.
x_scaler = StandardScaler().fit(X_train)
X_train = x_scaler.transform(X_train).astype('float32')
X_test = x_scaler.transform(X_test).astype('float32')

# Целевую величину тоже масштабируем — это упрощает подбор темпа обучения.
y_mean, y_std = y_train.mean(), y_train.std()
y_train_s = (y_train - y_mean) / y_std
y_test_s = (y_test - y_mean) / y_std

print(f'Обучающих примеров: {len(X_train)}, тестовых: {len(X_test)}')
print(f'Число входных признаков: {X_train.shape[1]}')

## 4. Применение метода

**Шаг 4.1. Архитектура сети.**

Строим небольшую полносвязную сеть — **многослойный перцептрон**:
- **Входной слой**: 10 нейронов (по числу признаков).
- **Первый скрытый слой**: 32 нейрона + функция активации ReLU. ReLU «включает» нейрон, если его взвешенная сумма входов положительна, и «выключает» иначе — это позволяет сети учить нелинейные зависимости.
- **Dropout (0.1)**: во время обучения случайно «отключает» 10 % нейронов — это снижает переобучение.
- **Второй скрытый слой**: ещё 32 нейрона + ReLU.
- **Выходной слой**: 1 нейрон — предсказываемое число.

Ячейка ниже определяет и выводит архитектуру сети.

In [ ]:
import torch.nn as nn


class MLPRegressor(nn.Module):
    """Полносвязная сеть для регрессии на табличных данных."""

    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = MLPRegressor(X_train.shape[1])
print(model)

**Шаг 4.2. Цикл обучения.**

Ячейка ниже запускает обучение на 120 эпохах. На каждой эпохе:
1. Данные перемешиваются и разбиваются на мини-пакеты (batch) по 32 наблюдения.
2. Для каждого пакета: сеть делает предсказание, вычисляется потеря (MSE — среднеквадратичная ошибка), затем методом обратного распространения ошибки вычисляются градиенты — насколько нужно изменить каждый вес.
3. Оптимизатор Adam обновляет веса по градиентам с учётом истории предыдущих обновлений.

После каждой эпохи записываем потерю на обучении и на тесте.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(torch.from_numpy(X_train),
                         torch.from_numpy(y_train_s))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

Xte = torch.from_numpy(X_test)
yte = torch.from_numpy(y_test_s)

n_epochs = 120
history = {'train': [], 'test': []}
for epoch in range(1, n_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        tr = criterion(model(torch.from_numpy(X_train)),
                       torch.from_numpy(y_train_s)).item()
        te = criterion(model(Xte), yte).item()
    history['train'].append(tr)
    history['test'].append(te)
    if epoch % 30 == 0 or epoch == 1:
        print(f'Эпоха {epoch:3d}: потеря обучение {tr:.4f}, тест {te:.4f}')

### Шаг 4.3. Кривые обучения и качество предсказаний

Ячейка ниже строит два графика:

1. **Кривые обучения** — потеря (MSE) по эпохам на обучающей и тестовой выборках. Помогает понять: учится ли сеть вообще и нет ли переобучения.
2. **Предсказание против истины** — каждая точка соответствует одному тестовому наблюдению.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error

model.eval()
with torch.no_grad():
    pred_s = model(Xte).numpy()
# Возврат к исходной шкале целевой величины.
pred = pred_s * y_std + y_mean

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
epochs = range(1, n_epochs + 1)
axes[0].plot(epochs, history['train'], color=VIZ['series'][0],
             label='Обучение')
axes[0].plot(epochs, history['test'], color=VIZ['series'][2],
             label='Проверка')
axes[0].set_title('Кривые обучения')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Среднеквадратичная потеря')
axes[0].legend(loc='upper right')

axes[1].scatter(y_test, pred, color=VIZ['series'][1], alpha=0.7,
                edgecolor='white', linewidth=0.5)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[1].plot(lims, lims, color=VIZ['series'][3], linestyle='--',
             label='Идеальное совпадение')
axes[1].set_title(f'Предсказание против истины (R2 = {r2:.3f})')
axes[1].set_xlabel('Истинное значение')
axes[1].set_ylabel('Предсказание сети')
axes[1].legend(loc='upper left')

fig.tight_layout()
plt.show()
print(f'Коэффициент детерминации R2: {r2:.3f}')
print(f'Средняя абсолютная ошибка: {mae:.1f}')

**Как читать эти графики:**

- **Кривые обучения (левый)**: обе кривые должны снижаться. Нормальная картина: кривая обучения (синяя) чуть ниже тестовой (оранжевой) — небольшой зазор допустим. Если тестовая кривая начинает расти, а обучающая продолжает падать — это **переобучение**: сеть запоминает обучающие данные, а не выучивает закономерность.

- **Предсказание против истины (правый)**: точки должны лежать вдоль пунктирной диагонали. Вертикальный разброс — типичная ошибка. Систематическое отклонение вверх или вниз — смещение модели в части диапазона. R² показывает долю объяснённой дисперсии: 0.5 означает, что модель объясняет половину разброса ответов.

## 5. Интерпретация результата

- **Кривые обучения**: обе линии должны снижаться. Если потеря на проверке растёт, а на обучении продолжает падать — это переобучение; помогают Dropout, уменьшение сети и ранняя остановка.
- **Коэффициент детерминации R2** показывает долю объяснённой дисперсии: 1.0 — идеально, 0.0 — не лучше среднего.
- **График «предсказание против истины»**: точки должны лежать вдоль диагонали. Систематическое отклонение указывает на смещение модели.
- **Средняя абсолютная ошибка** выражена в единицах целевой величины и легко интерпретируется содержательно.

Помните: для нейросети обязательны нормировка входов и оценка на независимой выборке. На небольших табличных данных стоит сравнить результат с градиентным бустингом — он часто точнее и проще в настройке.

## 5б. Наглядный эксперимент: переобучение на синтетических данных

Покажем переобучение наглядно: обучим маленькую и большую сеть на небольшом синтетическом наборе с нелинейной зависимостью. Маленькая сеть — недообучение (не уловила форму). Большая без регуляризации — переобучение (запомнила шум). Большая с Dropout — баланс.

Этот эксперимент занимает несколько секунд и не требует дополнительных пакетов.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# Синтетические данные: нелинейная зависимость + шум, мало точек.
n = 60
Xs = np.linspace(-3, 3, n).astype("float32")
ys = (np.sin(Xs) + 0.4 * np.cos(2 * Xs) + np.random.normal(0, 0.3, n)).astype("float32")
Xt = torch.from_numpy(Xs.reshape(-1, 1))
yt = torch.from_numpy(ys)
X_grid = torch.linspace(-3, 3, 300).reshape(-1, 1)


def make_net(hidden, dropout_p=0.0):
    layers = [nn.Linear(1, hidden), nn.ReLU()]
    if dropout_p > 0:
        layers.append(nn.Dropout(dropout_p))
    layers += [nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1)]
    return nn.Sequential(*layers)


def train_net(net, n_epochs=800, lr=1e-2):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(n_epochs):
        net.train()
        opt.zero_grad()
        loss_fn(net(Xt).squeeze(), yt).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return net(X_grid).squeeze().numpy()


configs = [
    ("Маленькая сеть (4 нейрона)\nНедообучение", make_net(4)),
    ("Большая сеть (256 нейронов)\nбез Dropout — переобучение", make_net(256)),
    ("Большая сеть (256 нейронов)\nDropout 0.3 — баланс", make_net(256, dropout_p=0.3)),
]
palette = get_palette(4)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

for ax, (title, net), color in zip(axes, configs, palette):
    preds = train_net(net)
    ax.scatter(Xs, ys, s=20, alpha=0.6, color=palette[3], zorder=3, label="Данные")
    ax.plot(X_grid.numpy(), preds, color=color, linewidth=2.2, label="Модель")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Признак")
    ax.legend(fontsize=9, loc="upper right")

axes[0].set_ylabel("Целевая величина")
fig.suptitle("Недообучение, переобучение и регуляризация Dropout", y=1.01, fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график:**

Три панели — три варианта сети на одних и тех же данных:

- **Левый (недообучение)**: кривая слишком прямая — сеть слишком мала, чтобы уловить форму зависимости.
- **Средний (переобучение)**: кривая «виляет» через каждую точку — сеть запомнила шум вместо закономерности. На новых данных такая модель ошибётся.
- **Правый (баланс с Dropout)**: кривая следует основной тенденции, но не воспроизводит каждый шум. Dropout «заставляет» разные нейроны учиться независимо и снижает переобучение.

Этот эффект — один из центральных в практике нейронных сетей: нужная сложность сети зависит от объёма данных.

## 6. Попробуйте на своих данных

Подготовьте таблицу: матрицу признаков `X` формы `(наблюдение, признак)` и вектор целевой величины `y`.

1. Загрузите CSV-файл в Colab через панель «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имя целевого столбца.
3. Обязательно нормируйте признаки и при необходимости целевую величину, как в разделе 3.
4. Последовательно выполните ячейки разделов 4 и 5.

## Поэкспериментируйте сами

На основном блокноте (раздел 4) попробуйте изменить параметры и перезапустить обучение:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `lr` в оптимизаторе | Попробуйте 0.1 и 0.0001 | При большом lr — кривая «прыгает»; при малом — сходится слишком медленно |
| `hidden=32` в `MLPRegressor` | Измените на 4 и на 256 | Как меняются кривые обучения? Появляется ли переобучение? |
| `n_epochs=120` | Увеличьте до 500 без Dropout | Разойдутся ли кривые обучения и теста? |
| `Dropout(0.1)` | Уберите слой Dropout | Изменится ли зазор между кривыми? |

На синтетическом примере (раздел 5б) попробуйте увеличить `n=200` — исчезнет ли переобучение у большой сети без Dropout при большем числе данных?

## Краткие выводы

- Нейронная сеть учится по примерам: многократно видит пары «вход–ответ» и корректирует свои веса методом обратного распространения ошибки.
- Нормировка входных признаков обязательна: без неё обучение нестабильно.
- Кривые обучения — главный инструмент диагностики: снижаются обе — всё хорошо; тестовая растёт — переобучение.
- Dropout снижает переобучение; при малых данных нужна небольшая сеть.
- Темп обучения (`learning_rate`) — критический параметр: начните с 1e-3 или используйте стандартный Adam.
- На небольших табличных данных градиентный бустинг часто точнее нейросети и проще в настройке — всегда сравнивайте.
- Нейронная сеть — универсальный аппроксиматор, но для специализированных данных (изображения, тексты, последовательности) используйте профильные архитектуры.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике кривых обучения потеря на обучающей выборке к 120-й эпохе составляет 0.08, а на тестовой — 0.31 и продолжает расти. Как называется это явление, какова его причина в данном контексте, и какие два изменения в архитектуре или процедуре обучения вы попробуете первыми?
2. Вы убрали `StandardScaler` из пайплайна и заново обучили сеть — потери скачут хаотично и не сходятся. Объясните механизм: почему отсутствие нормировки входов нарушает обучение нейронной сети, тогда как, например, случайному лесу она не нужна?
3. Вы хотите ускорить обучение и задаёте `lr=0.5` вместо `5e-3`. Кривая потерь начинает резко прыгать вверх-вниз и не убывает. Что происходит с шагом обновления весов, и как найти рабочий диапазон темпа обучения без полного перебора?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это переобучение: модель запомнила шум обучающей выборки вместо общей закономерности. Первые шаги: (а) увеличить коэффициент Dropout (например, с 0.1 до 0.3–0.5) — это снижает зависимость нейронов друг от друга; (б) уменьшить число нейронов в скрытых слоях (`hidden` с 32 до 16), чтобы снизить ёмкость модели. Дополнительно можно применить ранннюю остановку по тестовой потере.
2. Нейронные сети суммируют входы с весами: если один признак имеет масштаб 1000, а другой — 0.001, градиенты для их весов будут отличаться на 6 порядков. Это приводит к «взрывным» или «затухающим» градиентам, неустойчивому ландшафту потерь и медленной или нестабильной сходимости. Решающее дерево делит по порогу — абсолютные значения не важны, только порядок; поэтому масштаб не влияет.
3. При слишком большом темпе обучения шаг обновления весов превышает кривизну ландшафта потерь, и оптимизатор «перепрыгивает» через минимум. Рабочий диапазон ищут методом learning rate finder: запускают обучение, каждую итерацию увеличивая lr в геометрической прогрессии, и находят точку, где потеря начинает расти. Значение на одну-две декады меньше этой точки — обычно хороший выбор.
</details>

In [ ]:
# --- Шаблон загрузки своих табличных данных ---
# import pandas as pd
#
# df = pd.read_csv('my_data.csv')
# target = 'имя_целевого_столбца'
# X = df.drop(columns=[target]).to_numpy(dtype='float32')
# y = df[target].to_numpy(dtype='float32')
#
# print(f'Объектов: {len(X)}, признаков: {X.shape[1]}')
# Далее повторите шаги нормировки и обучения из разделов 3-4.